# 001 · itertools Essentials

> **Standard Library › Functional Tools** · Python 3.14

`itertools` is Python's toolbox of fast, **lazy** building blocks for working with iterables. If you think in LINQ — `Skip`, `Take`, `Concat`, `SelectMany`, `Chunk`, `Zip` — this is the module where most of those operators live.

**In this notebook you will learn:**

- how the common LINQ operators map to `itertools` functions
- slicing and filtering streams: `islice`, `takewhile`, `dropwhile`, `filterfalse`, `compress`
- combining streams: `chain`, `chain.from_iterable`, `zip_longest`, `pairwise`, `batched`
- infinite generators: `count`, `cycle`, `repeat`
- why everything is lazy, and the "an iterator can only be consumed once" gotcha

## .NET mental model

`itertools` ≈ `System.Linq.Enumerable`, but as **free functions** that take iterables, rather than extension methods you chain with dots.

| LINQ (`System.Linq`) | Python |
|---|---|
| `Skip(n)` / `Take(n)` | `itertools.islice(it, n, None)` / `islice(it, n)` |
| `TakeWhile` / `SkipWhile` | `takewhile` / `dropwhile` |
| `Where(x => !pred(x))` | `filterfalse(pred, it)` |
| `Concat` | `chain(a, b, ...)` |
| `SelectMany(x => x)` | `chain.from_iterable(nested)` |
| `Enumerable.Range(start, count)` | `range(start, stop)` (finite) or `count(start)` (infinite) |
| `Enumerable.Repeat(x, n)` | `repeat(x, n)` |
| `Chunk(n)` (.NET 6+) | `batched(it, n)` (3.12+) |
| `Zip` | built-in `zip` (shortest) / `zip_longest` |
| `Select((a, b) => f(a, b))` over tuples | `starmap(f, pairs)` |
| `Select` / `Where` | built-in `map` / `filter`, or a generator expression |

Like LINQ, every `itertools` function is **lazy** (deferred execution): nothing is computed until you iterate. Unlike LINQ, the result is an *iterator* — it can be consumed **only once**.

## Laziness: nothing happens until you iterate

```csharp
// C# — deferred execution
var query = numbers.Where(n => n > 2).Select(n => n * 10);
var list  = query.ToList();   // executes here
```

In Python, `itertools` functions return iterator objects. Wrap them in `list()` (≈ `ToList()`) to materialise, or loop over them.

In [ ]:
import itertools as it

numbers = [1, 2, 3, 4, 5]
query = it.islice(numbers, 2)     # nothing computed yet
print(type(query).__name__)       # 'islice' — an iterator object, not a list
print(list(query))                # [1, 2]  — like ToList()

⚠️ **Gotcha:** an iterator is a one-shot stream. A LINQ `IEnumerable<T>` query can be enumerated again (it re-runs), but a Python iterator is **exhausted** after one pass and silently yields nothing the second time *(see Syntax › Iterators & Generators › 001 · Iterables and Iterators)*.

In [ ]:
evens = filter(lambda n: n % 2 == 0, range(10))
print(list(evens))   # [0, 2, 4, 6, 8]
print(list(evens))   # []  — already consumed, no error!

**Tip:** if you need the data twice, materialise it once with `list(...)`, or recreate the iterator. (`itertools.tee` exists too — see *Standard Library › Functional Tools › 002 · itertools: Combinatorics and Grouping*.)

## `islice` — Skip and Take

`islice(iterable, stop)` or `islice(iterable, start, stop[, step])` slices **any** iterable — including generators and files that don't support `[a:b]`.

```csharp
var page = items.Skip(20).Take(10);
var firstThree = items.Take(3);
```

In [ ]:
from itertools import islice

letters = "abcdefghij"
print(list(islice(letters, 3)))           # Take(3)          -> a b c
print(list(islice(letters, 3, None)))     # Skip(3)          -> d..j
print(list(islice(letters, 2, 6)))        # Skip(2).Take(4)  -> c d e f
print(list(islice(letters, 0, None, 3)))  # every 3rd item   -> a d g j

The big win is slicing something you **can't** index — such as an infinite generator:

In [ ]:
def squares():
    n = 0
    while True:          # infinite!
        yield n * n
        n += 1

print(list(islice(squares(), 5, 10)))   # Skip(5).Take(5)

⚠️ **Gotcha:** `islice` does not accept negative indexes (there is no `TakeLast`) — a stream has no known end. Use a `list` and normal slicing, or `collections.deque(it, maxlen=n)` for "last *n*".

In [ ]:
try:
    islice(letters, -3, None)
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

## `takewhile` and `dropwhile`

Identical semantics to LINQ's `TakeWhile` / `SkipWhile`: they look at items **from the start** and stop checking once the predicate first fails.

```csharp
var warmUp = readings.TakeWhile(r => r < 50);
var rest   = readings.SkipWhile(r => r < 50);
```

In [ ]:
from itertools import takewhile, dropwhile

readings = [10, 20, 35, 60, 15, 80]
print(list(takewhile(lambda r: r < 50, readings)))   # [10, 20, 35]
print(list(dropwhile(lambda r: r < 50, readings)))   # [60, 15, 80]  — 15 kept!

Note the argument order: the **function comes first**, then the iterable. This is true for almost every function in `itertools` (and for built-in `map` / `filter`).

## `filterfalse` and `compress`

- `filterfalse(pred, it)` keeps the items where `pred` is falsy — the inverse of built-in `filter`.
- `compress(data, selectors)` keeps the items whose matching selector is truthy — handy when the mask comes from elsewhere.

In [ ]:
from itertools import filterfalse, compress

nums = range(10)
print(list(filter(lambda n: n % 3 == 0, nums)))       # [0, 3, 6, 9]
print(list(filterfalse(lambda n: n % 3 == 0, nums)))  # everything else

products = ["pen", "desk", "lamp", "chair"]
in_stock = [True, False, True, False]
print(list(compress(products, in_stock)))             # ['pen', 'lamp']

## `chain` — Concat and SelectMany

```csharp
var all  = first.Concat(second).Concat(third);
var flat = orders.SelectMany(o => o.Lines);   // flatten nested sequences
```

`chain(a, b, c)` concatenates a fixed number of iterables. `chain.from_iterable(nested)` flattens one level — the lazy equivalent of `SelectMany(x => x)`.

In [ ]:
from itertools import chain

first, second = [1, 2], (3, 4)
print(list(chain(first, second, "ab")))     # any iterables, mixed types: [1, 2, 3, 4, 'a', 'b']

nested = [[1, 2], [3], [], [4, 5, 6]]
print(list(chain.from_iterable(nested)))    # [1, 2, 3, 4, 5, 6]

`SelectMany` with a selector is `chain.from_iterable` over a generator expression (or simply a nested comprehension):

In [ ]:
orders = [
    {"id": 1, "lines": ["pen", "ink"]},
    {"id": 2, "lines": ["desk"]},
]

# orders.SelectMany(o => o.Lines)
print(list(chain.from_iterable(o["lines"] for o in orders)))
print([line for o in orders for line in o["lines"]])   # same, eagerly

## Infinite iterators: `count`, `cycle`, `repeat`

| Python | Produces | .NET analogue |
|---|---|---|
| `count(start=0, step=1)` | `start, start+step, ...` forever | `Enumerable.Range` without an end |
| `cycle(it)` | items of `it`, repeated forever | no BCL equivalent |
| `repeat(x)` / `repeat(x, n)` | `x` forever / `n` times | `Enumerable.Repeat(x, n)` |

⚠️ Never call `list()` on an infinite iterator — always bound it with `islice`, `zip`, `takewhile`, or a `break`.

In [ ]:
from itertools import count, cycle, repeat

print(list(islice(count(10), 5)))          # 10 11 12 13 14
print(list(islice(count(0, 0.25), 5)))     # floats work too
print(list(islice(cycle("RGB"), 7)))       # R G B R G B R
print(list(repeat("x", 3)))                # Enumerable.Repeat("x", 3)

A typical use: `zip` stops at the shortest input, so an infinite iterator pairs nicely with a finite one.

In [ ]:
tasks = ["build", "test", "deploy", "notify"]
workers = cycle(["alice", "bob"])          # round-robin assignment

for ticket_id, task, worker in zip(count(100), tasks, workers):
    print(ticket_id, task, worker)

`repeat` is also a fast constant argument for `map`:

In [ ]:
print(list(map(pow, [2, 3, 4], repeat(2))))   # squares: [4, 9, 16]

## `batched` — Chunk (3.12+)

```csharp
foreach (int[] chunk in Enumerable.Range(1, 7).Chunk(3))
    Console.WriteLine(string.Join(",", chunk));   // 1,2,3 / 4,5,6 / 7
```

`itertools.batched(it, n)` (added in 3.12) yields **tuples** of up to `n` items; the last one may be shorter, exactly like `Chunk`.

In [ ]:
from itertools import batched

for batch in batched(range(1, 8), 3):
    print(batch)

Since 3.13 you can pass `strict=True` to demand that every batch is full — handy when an incomplete batch means corrupt data:

In [ ]:
try:
    list(batched(range(1, 8), 3, strict=True))
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

## `pairwise` — sliding window of two (3.10+)

LINQ has no built-in for this; the usual C# trick is `seq.Zip(seq.Skip(1))`. Python's `pairwise` returns overlapping pairs `(s0, s1), (s1, s2), ...`.

```csharp
var deltas = prices.Zip(prices.Skip(1), (a, b) => b - a);
```

In [ ]:
from itertools import pairwise

prices = [100, 104, 101, 107]
print(list(pairwise(prices)))                    # [(100, 104), (104, 101), (101, 107)]
print([b - a for a, b in pairwise(prices)])      # deltas: [4, -3, 6]

## `zip_longest` — zip without truncation

Built-in `zip` (like LINQ's `Zip`) stops at the shortest input. `zip_longest` keeps going and pads with `fillvalue` (default `None`).

In [ ]:
from itertools import zip_longest

names = ["alice", "bob", "carol"]
scores = [90, 85]
print(list(zip(names, scores)))                       # carol dropped
print(list(zip_longest(names, scores)))               # padded with None
print(list(zip_longest(names, scores, fillvalue=0)))  # padded with 0

**Tip:** if the inputs *should* have equal length, use built-in `zip(a, b, strict=True)` (3.10+) instead — it raises `ValueError` on a mismatch rather than silently dropping data.

In [ ]:
try:
    list(zip(names, scores, strict=True))
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

## `starmap` — map over argument tuples

`map(f, xs)` calls `f(x)`. `starmap(f, pairs)` calls `f(*pair)` — it unpacks each tuple into positional arguments. Think `Select(t => f(t.Item1, t.Item2))`.

In [ ]:
from itertools import starmap

pairs = [(2, 5), (3, 2), (10, 3)]
print(list(starmap(pow, pairs)))                   # [32, 9, 1000]
print([pow(base, exp) for base, exp in pairs])     # same with a comprehension

## Putting it together: a lazy pipeline

Because every stage is lazy, you can process a huge (or infinite) stream with constant memory — exactly like a LINQ query over an `IEnumerable<T>`.

In [ ]:
def sensor_stream():
    """Pretend this reads forever from a device."""
    values = [3, 7, 0, 12, 5, 0, 9, 14, 2, 8, 11, 6]
    yield from cycle(values)

readings = sensor_stream()
non_zero = filterfalse(lambda v: v == 0, readings)   # Where(v => v != 0)
first_ten = islice(non_zero, 10)                     # Take(10)
for batch in batched(first_ten, 4):                  # Chunk(4)
    print(batch, "avg =", sum(batch) / len(batch))

⚠️ **Gotcha:** functions like `len()` don't work on iterators — there is no `Count()` that consumes the stream for you. Use `sum(1 for _ in it)` if you really need to count (and remember that consumes it).

In [ ]:
stream = islice(count(), 5)
try:
    len(stream)
except TypeError as e:
    print(f"{type(e).__name__}: {e}")
print(sum(1 for _ in stream))   # 5 — and now stream is exhausted

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Take first *n* | `.Take(n)` | `islice(it, n)` |
| Skip first *n* | `.Skip(n)` | `islice(it, n, None)` |
| Take/skip while | `.TakeWhile(p)` / `.SkipWhile(p)` | `takewhile(p, it)` / `dropwhile(p, it)` |
| Inverse filter | `.Where(x => !p(x))` | `filterfalse(p, it)` |
| Filter by mask | `Zip` + `Where` | `compress(data, mask)` |
| Concatenate | `.Concat(b)` | `chain(a, b)` |
| Flatten | `.SelectMany(x => x)` | `chain.from_iterable(nested)` |
| Counter / range | `Enumerable.Range(s, n)` | `count(s)` / `range(s, s + n)` |
| Repeat | `Enumerable.Repeat(x, n)` | `repeat(x, n)` |
| Round-robin | — | `cycle(it)` |
| Chunk | `.Chunk(n)` | `batched(it, n)` (3.12+) |
| Adjacent pairs | `s.Zip(s.Skip(1))` | `pairwise(s)` (3.10+) |
| Zip, pad shorter | — | `zip_longest(a, b, fillvalue=...)` |
| Apply to tuples | `.Select(t => f(t.A, t.B))` | `starmap(f, pairs)` |
| Materialise | `.ToList()` | `list(it)` |

## Exercises

1. Given `log = ["# header", "# version 2", "start", "# note", "stop"]`, skip the leading comment lines (only the leading ones!) and print the rest.
2. Flatten `[[1, 2], [3], [4, 5]]` and print the items in batches of 2.
3. Using `count` and `islice`, print the first 5 multiples of 7 that are greater than 50.
4. Given `temps = [18, 21, 19, 24, 22]`, print each day-over-day change as `"18 -> 21: +3"`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
from itertools import dropwhile, chain, batched, count, islice, pairwise

# 1 — dropwhile stops checking after the first non-comment line
log = ["# header", "# version 2", "start", "# note", "stop"]
print(list(dropwhile(lambda line: line.startswith("#"), log)))
# ['start', '# note', 'stop']

# 2 — chain.from_iterable is SelectMany; batched is Chunk
for pair in batched(chain.from_iterable([[1, 2], [3], [4, 5]]), 2):
    print(pair)

# 3 — an infinite count, filtered lazily, then bounded by islice
multiples = (n for n in count(7, 7) if n > 50)
print(list(islice(multiples, 5)))   # [56, 63, 70, 77, 84]

# 4 — pairwise gives overlapping (previous, current) pairs
temps = [18, 21, 19, 24, 22]
for a, b in pairwise(temps):
    print(f"{a} -> {b}: {b - a:+d}")
```

</details>